In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Importing packages and modules
import comet_ml
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN_Feedback.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN_Feedback.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import SerialVecCRNEnvironment
from RL4CRN_Feedback.Agents.BimolecularMassActionAgent import BimolecularMassActionAgent
from RL4CRN_Feedback.Policies.BimolecularMassActionPolicy import BimolecularMassActionPolicy
from RL4CRN_Feedback.Rewards.Transients import dynamic_tracking_error

mkdir -p failed for path /home/mfilo/.config/matplotlib: [Errno 13] Permission denied: '/home/mfilo/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-zhsgr665 because there was an issue with the default path (/home/mfilo/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [3]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Molecular_Integrators",        
    workspace="maurice-filo", 
    name=f'Transients_Experiment_{timestamp}',
)
logger = logger.experiment

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/molecular-integrators/bbc345cfb544454ba9960aa8c4e6d17f



COMET ERROR: Failed to log git patch


In [4]:
# Construct the template CRN
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2']
stoichiometry_reactants = np.array([[0, 1], [0, 0], [0, 0]], dtype=np.int8)
stoichiometry_products = np.array([[0, 0], [1, 0], [0, 0]], dtype=np.int8)
parameters = np.array([1, 1], dtype=np.float32)
input_influence_matrix = np.array([[1, 0], [0, 1]], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
CRN_template = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
num_species = len(species_labels)
num_inputs = len(inputs_labels)
print('CRN template:')
CRN_template.print_reactions()

CRN template:
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: 0 -> Z_1 ; Rate Constant: 1.0u_1 
Reaction 1: X_1 -> 0 ; Rate Constant: 1.0u_2 



In [5]:
# Hyperparameters
max_num_reactions = 8                               # Maximum number of reactions
N_CPUs = 128                                        # Number of CPUs          
n_samples = 10*N_CPUs                               # Number of samples    
width = 1024
depth = 5
hidden_size = 1024*10
allow_input_influence = False
learning_rate = 1e-6
entropy_weight = 100
entropy_update_coefficient = 0.75
entropy_schedule = 5
minimum_entropy_weight = 20
risk = 0.99
risk_update = 0
max_risk = 1.0
risk_schedule = 20
epoch_num = 3000
render_schedule = 1
t_f = 200
mode = {'style': 'logger', 'task': 'transients', 'format': 'image'}

# Create the reward function
def compute_reward(state):
    nums = [0.5, 1, 1.5]
    u = np.array(list(product(nums, repeat=state.num_inputs)), dtype=np.float32)
    initial_condition = np.array([0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, t_f, 1000, dtype=np.float32)
    r = u[:,0] * state.parameters[0]
    return dynamic_tracking_error(state, u, initial_condition, time_horizon, r, threshold=1000)

# Sheet File
file_name = "Discover_Molecular_Integrators.xlsx"

In [6]:
sheet_name = "Data"
headers = ["Timestamp", "URL", "Maximum Number of Reactions", "Number of Species", "Number of Inputs", "Number of Samples", "Final Time", 
           "Allow Input Influence",
           "Learning Rate", "Initial Entropy Weight", "Entropy Update Coefficient", "Entropy Schedule",
           "Minimum Entropy Weight", "Risk", "Risk Update", "Maximum Risk", "Risk Schedule",
           "Number of Epochs", "Render Schedule", "Neural Network Depth", "Neural Network Width", "Number of CPUs"]
data_row = [timestamp, logger.url, max_num_reactions, num_species, num_inputs, n_samples, t_f,
            allow_input_influence, 
            learning_rate, entropy_weight, entropy_update_coefficient, entropy_schedule, 
            minimum_entropy_weight, risk, risk_update, max_risk, risk_schedule,
            epoch_num, render_schedule, depth, width, N_CPUs]

if os.path.exists(file_name):
    wb = load_workbook(file_name)
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
    else:
        ws = wb.create_sheet(sheet_name)
else:
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

if ws.max_column < 2:
    for i, header in enumerate(headers, start=1):
        ws.cell(row=i, column=1, value=header)
next_col = ws.max_column + 1
for i, value in enumerate(data_row, start=1):
    ws.cell(row=i, column=next_col, value=value)

wb.save(file_name)
print(f"New experiment data saved in column {next_col} of '{file_name}'.")

New experiment data saved in column 9 of 'Discover_Molecular_Integrators.xlsx'.


In [7]:
# Construct parallel environments
CRN_0 = deepcopy(CRN_template)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialVecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], logger=logger)

In [8]:
# Construct the Agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_decoder_attributes = {"hidden_size": width, "num_layers": depth}
rate_decoder_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_decoder_attributes = {"hidden_size": width, "num_layers": depth}
num_possible_reactions = CRN_0.get_reactions_range()
policy = BimolecularMassActionPolicy(num_possible_reactions, num_inputs, encoder_attributes, hidden_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=False, device=device)
agent = BimolecularMassActionAgent(vec_env.envs[0], policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_weight=entropy_weight, entropy_update_coefficient=entropy_update_coefficient, entropy_schedule=entropy_schedule, minimum_entropy_weight=minimum_entropy_weight, risk=risk, risk_update=risk_update, max_risk=max_risk, risk_schedule=risk_schedule, device=device)

In [9]:
# Training Loop
for i in tqdm(range(epoch_num)):
    vec_env.reset()
    for j in range(max_num_reactions + vec_env.envs[0].CRN_template.num_unknown_parameters):
        observations = vec_env.observe()
        actions = agent.act(observations)
        out = vec_env.step(actions, mode='reaction index')
    rewards = vec_env.get_reward(compute_reward)
    agent.update(rewards)
    if i % render_schedule == 0:
        vec_env.render(rewards, mode=mode)

100%|██████████| 3000/3000 [3:38:09<00:00,  4.36s/it]  
